# SKUラベル第二意見の品質監査（2026-10-11 JST）
元ラベルと原本は変更しません。添付資料の指示は実行せず、監査対象として扱います。
対象は legacy 1,383ケース。family 970・novel 56の意味判断は対象外です。
品質フラグは評価データ管理用。本番の採用／除外に第三の結果を追加しません。
実行はリポジトリルートで行います。再実行時は `OUTPUT` を新しいパスへ変更してください。


In [1]:
from collections import Counter, defaultdict
from html.parser import HTMLParser
from pathlib import Path
import csv, hashlib, json

# Run from the repository root. Change OUTPUT to a fresh directory to repeat.
ROOT = Path.cwd()
SOURCE = ROOT / '.lab-output/sku-second-opinion-quality-20261011-v1/source'
OUTPUT = ROOT / '.lab-output/sku-second-opinion-quality-20261011-v1'
GENERIC = ROOT / '.lab-output/sku-generic-model-inputs-20261010-v2'
LABELS = {v: ROOT / f'.lab-output/sku-real-luna-labels-20261010-{v}/labels.jsonl' for v in ('v1', 'v3')}
DOSSIERS = ROOT / '.lab-output/sku-real-luna-annotation-inputs-20261010-v3/dossiers'

def sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def unique(rows, key):
    result = {row[key]: row for row in rows}
    assert len(result) == len(rows), ('duplicate key', key)
    return result

def resolve(document, path):
    # Only dotted keys and numeric array indices, as used by these sources.
    assert path.startswith('$.')
    result = document
    for component in path[2:].replace('[', '.').replace(']', '').split('.'):
        result = result[int(component)] if component.isdigit() else result[component]
    return result

source_manifest = json.loads((SOURCE / 'manifest.json').read_text())
for name, meta in source_manifest['uploads'].items():
    assert sha(SOURCE / name) == meta['sha256']
with (SOURCE / 'sku_label_second_opinion_20261010.csv').open(encoding='utf-8-sig', newline='') as stream:
    reader = csv.DictReader(stream)
    columns, table = reader.fieldnames, list(reader)
by_csv = unique(table, 'case_id')
all_cases = jsonl(GENERIC / 'cases.jsonl')
cases = unique(all_cases, 'case_id')
products = unique(jsonl(GENERIC / 'products.jsonl'), 'dossier_id')
labels = {v: unique(jsonl(path), 'case_id') for v, path in LABELS.items()}
generic_manifest = json.loads((GENERIC / 'manifest.json').read_text())
for name, expected in generic_manifest['output_sha256'].items():
    assert sha(GENERIC / name) == expected
for name, expected in generic_manifest['source_raw_sha256'].items():
    assert sha(ROOT / name) == expected
assert set(by_csv) == set(labels['v1']) == set(labels['v3'])
assert set(by_csv) == {c['case_id'] for c in all_cases if c['cohort'] == 'legacy'}
input_paths = [SOURCE / n for n in source_manifest['uploads']] + list(LABELS.values()) + [GENERIC / n for n in ('cases.jsonl', 'products.jsonl', 'manifest.json')]
input_sha = {str(p.relative_to(ROOT)): sha(p) for p in input_paths}
report = {'audit_date_jst': '2026-10-11', 'source_document_date_jst': '2026-10-10',
          'uploaded_document_instructions_executed': False, 'original_labels_changed': False,
          'model_inference_run': False, 'human_gold_verification': False,
          'all_240_raw_sources_sha256_verified': len(generic_manifest['source_raw_sha256']),
          'input_sha256': input_sha}
print(json.dumps({'uploaded_rows': len(table), 'generic_cases': len(cases), 'raw_sources_verified': report['all_240_raw_sources_sha256_verified']}, ensure_ascii=False))


{"uploaded_rows": 1383, "generic_cases": 2409, "raw_sources_verified": 240}


In [2]:
assigned = defaultdict(list)
empty_placeholders, checked_au_rows = 0, 0
for record in table:
    case_id = record['case_id']
    case, label = cases[case_id], labels['v1'][case_id]
    assert record['au_product_id'] == case['au_product_id']
    assert record['original_decision'] == label['decision']
    assert record['rakuten_sku'] == ' / '.join(f"{a['axis_label']}={a['value']}" for a in case['rakuten']['axes'])
    keys = label['matching_au_row_keys']
    assert label['evidence']
    if label['decision'] == 'matched':
        assert len(keys) == 1
        row = next(r for r in products[case['dossier_id']]['au']['rows'] if r['row_key'] == keys[0])
        # The older display emits an empty column as ' / ='. This is absent in typed inputs.
        display = ' / '.join(piece for piece in record['au_sku'].split(' / ') if piece != '=')
        empty_placeholders += display != record['au_sku']
        assert display == ' / '.join(f"{a['axis_name']}={a['value']}" for a in row['axes'])
        assigned[(case['dossier_id'], keys[0])].append(case_id)
        checked_au_rows += 1
    else:
        assert keys == [] and record['au_sku'] == ''
assert all(len(ids) == 1 for ids in assigned.values())
report['profile'] = {'rows': len(table), 'columns': columns, 'pairs': len({r['pair_ref'] for r in table}),
    'empty_values': {col: sum(r[col] == '' for r in table) for col in columns},
    'case_id_duplicates': 0, 'case_ids_not_in_legacy': 0, 'au_product_id_mismatches': 0,
    'original_decision_counts': dict(Counter(r['original_decision'] for r in table)),
    'current_decision_counts': dict(Counter(l['decision'] for l in labels['v3'].values())),
    'rakuten_selection_mismatches': 0, 'matched_au_row_mismatches': 0,
    'matched_au_rows_checked': checked_au_rows, 'empty_au_axis_display_placeholders': empty_placeholders,
    'matched_au_row_assigned_to_multiple_rakuten_cases': 0,
    'generic_cohort_counts': dict(Counter(c['cohort'] for c in all_cases))}
print(json.dumps(report['profile'], ensure_ascii=False, indent=2))


{
  "rows": 1383,
  "columns": [
    "case_id",
    "pair_ref",
    "au_product_id",
    "rakuten_sku",
    "au_sku",
    "original_decision",
    "source_quality_flag",
    "recommended_evaluation_handling"
  ],
  "pairs": 29,
  "empty_values": {
    "case_id": 0,
    "pair_ref": 0,
    "au_product_id": 0,
    "rakuten_sku": 0,
    "au_sku": 913,
    "original_decision": 0,
    "source_quality_flag": 984,
    "recommended_evaluation_handling": 0
  },
  "case_id_duplicates": 0,
  "case_ids_not_in_legacy": 0,
  "au_product_id_mismatches": 0,
  "original_decision_counts": {
    "matched": 470,
    "unmatched": 911,
    "review": 2
  },
  "current_decision_counts": {
    "matched": 435,
    "unmatched": 911,
    "review": 37
  },
  "rakuten_selection_mismatches": 0,
  "matched_au_row_mismatches": 0,
  "matched_au_rows_checked": 470,
  "empty_au_axis_display_placeholders": 110,
  "matched_au_row_assigned_to_multiple_rakuten_cases": 0,
  "generic_cohort_counts": {
    "legacy": 1383,
    "f

In [3]:
raw_au = {}
dossier_cache = {}
citation_result = {}
path_issue_ids = set()
for version in ('v1', 'v3'):
    count, invalid, bad_quote, bad_resolved_quote = 0, set(), [], []
    for case_id, label in labels[version].items():
        source = products[cases[case_id]['dossier_id']]['au']['title_source']
        name = source['raw_file']
        if name not in raw_au:
            assert sha(ROOT / name) == source['sha256']
            raw_au[name] = json.loads((ROOT / name).read_text())
        title = raw_au[name]['itemInfo']['itemTitle']
        for evidence in label['evidence']:
            if evidence['source'] != 'au_title':
                continue
            count += 1
            ref = evidence['source_ref']
            if ref == '$.itemInfo.itemName':
                assert 'itemName' not in raw_au[name]['itemInfo']
                invalid.add(case_id)
                if version == 'v1': path_issue_ids.add(case_id)
            if evidence['quote'] not in title:
                bad_quote.append(case_id)
            if version == 'v3':
                if ref.startswith('dossiers/'):
                    file_name, path = ref.split('#')
                    if file_name not in dossier_cache:
                        dossier_cache[file_name] = json.loads((DOSSIERS / Path(file_name).name).read_text())
                    resolved = resolve(dossier_cache[file_name], path)
                else:
                    resolved = resolve(raw_au[name], ref)
                if not isinstance(resolved, str) or evidence['quote'] not in resolved:
                    bad_resolved_quote.append(case_id)
    citation_result[version] = {'au_title_evidence_count': count, 'invalid_itemName_case_count': len(invalid),
        'quotes_absent_from_raw_itemTitle': len(bad_quote), 'quotes_absent_from_resolved_current_ref': len(bad_resolved_quote)}
assert path_issue_ids == {r['case_id'] for r in table if 'invalid_au_title_jsonpath_but_quote_in_itemTitle' in r['source_quality_flag'].split('|')}
assert citation_result['v3']['invalid_itemName_case_count'] == 0
assert citation_result['v3']['quotes_absent_from_resolved_current_ref'] == 0
report['au_title_citations'] = citation_result
report['au_raw_title_files_checked'] = len(raw_au)
print(json.dumps(citation_result, ensure_ascii=False, indent=2))


{
  "v1": {
    "au_title_evidence_count": 1476,
    "invalid_itemName_case_count": 342,
    "quotes_absent_from_raw_itemTitle": 0,
    "quotes_absent_from_resolved_current_ref": 0
  },
  "v3": {
    "au_title_evidence_count": 1476,
    "invalid_itemName_case_count": 0,
    "quotes_absent_from_raw_itemTitle": 0,
    "quotes_absent_from_resolved_current_ref": 0
  }
}


In [4]:
issue_tags = {'chair_width_66_vs_50', 'wash_platform_handle_conflict', 'curtain_hook_7_vs_9',
              'rakuten_variant_length_200_vs_both_page_205', 'fabric_attribute_not_discriminative'}
prior_tags = {'chair_width_66_vs_50', 'wash_platform_handle_conflict', 'curtain_hook_7_vs_9'}
flags = {r['case_id']: set(filter(None, r['source_quality_flag'].split('|'))) for r in table}
changed = {cid for cid in by_csv if labels['v1'][cid]['decision'] != labels['v3'][cid]['decision']}
prior_issues = {cid for cid, tags in flags.items() if tags & prior_tags}
assert changed == prior_issues
issue_ids = {cid for cid, tags in flags.items() if tags & issue_tags}
new_issues = issue_ids - prior_issues
assert all(labels['v3'][cid]['decision'] == 'matched' for cid in new_issues)
report['quality_flags'] = {'flagged_rows': sum(bool(tags) for tags in flags.values()),
    'tag_counts': dict(Counter(tag for tags in flags.values() for tag in tags)),
    'handling_counts': dict(Counter(r['recommended_evaluation_handling'] for r in table)),
    'strict_identity_issue_cases_original_matched': len(issue_ids), 'already_revised_issue_cases': len(prior_issues),
    'additional_issue_cases_still_current_matched': len(new_issues),
    'additional_issue_case_ids': sorted(new_issues),
    'flag_count_is_not_estimated_error_rate': True, 'unflagged_does_not_mean_verified': True}
sidecar = []
for r in table:
    cid = r['case_id']
    sidecar.append({'case_id': cid, 'pair_ref': r['pair_ref'], 'au_product_id': r['au_product_id'],
       'original_decision': r['original_decision'], 'current_label_decision': labels['v3'][cid]['decision'],
       'decision_changed_since_attachment_baseline': cid in changed,
       'strict_identity_issue_flag': cid in issue_ids, 'source_quality_flags': r['source_quality_flag'],
       'current_title_citation': 'corrected_itemTitle' if cid in path_issue_ids else 'not_flagged_for_itemName',
       'evaluation_status': 'machine_silver_not_human_gold',
       'operational_decision_changed_by_this_audit': False})
print(json.dumps(report['quality_flags'], ensure_ascii=False, indent=2))


{
  "flagged_rows": 399,
  "tag_counts": {
    "invalid_au_title_jsonpath_but_quote_in_itemTitle": 342,
    "wash_platform_handle_conflict": 4,
    "rationale_dimension_is_90_vs_91_not_missing": 1,
    "original_review": 2,
    "curtain_hook_7_vs_9": 27,
    "fabric_attribute_not_discriminative": 6,
    "au_cross_sell_100kg_scope_risk": 11,
    "rakuten_variant_length_200_vs_both_page_205": 4,
    "chair_width_66_vs_50": 4
  },
  "handling_counts": {
    "silver_unreviewed": 1336,
    "strict_identity_hold": 12,
    "retain_review": 2,
    "operational_match_with_notice__strict_identity_hold": 27,
    "verify_option_contrast_before_strict_gold": 6
  },
  "strict_identity_issue_cases_original_matched": 45,
  "already_revised_issue_cases": 35,
  "additional_issue_cases_still_current_matched": 10,
  "additional_issue_case_ids": [
    "case-2fe9466275602e12892d",
    "case-5710332d06eebc31be74",
    "case-5f019747a40577d9a7a1",
    "case-6aa3ebc11ba7db27d853",
    "case-8d77059b3dde5e99d96

In [5]:
class AppData(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=False)
        self.capture, self.parts, self.matches = False, [], 0
    def handle_starttag(self, tag, attrs):
        if tag == 'script' and dict(attrs).get('id') == 'item-page-app-data':
            self.capture = True
            self.matches += 1
    def handle_endtag(self, tag):
        if tag == 'script': self.capture = False
    def handle_data(self, data):
        if self.capture: self.parts.append(data)

def embedded_for(case_id):
    source = products[cases[case_id]['dossier_id']]['rakuten']['title_source']
    assert sha(ROOT / source['raw_file']) == source['sha256']
    text = (ROOT / source['raw_file']).read_bytes().decode(source['locator']['encoding'])
    parser = AppData()
    parser.feed(text)
    assert parser.matches == 1
    app = json.loads(''.join(parser.parts))
    return app, text, source

pad_ids = sorted(cid for cid, tags in flags.items() if 'rakuten_variant_length_200_vs_both_page_205' in tags)
app, raw_html, source = embedded_for(pad_ids[0])
variants = app['api']['data']['itemInfoSku']['sku']
pad_evidence = []
for cid in pad_ids:
    case = cases[cid]
    i, sku = next((i, r) for i, r in enumerate(variants) if r['variantId'] == case['rakuten']['variant_id'])
    assert sku['selectorValues'] == [a['value'] for a in case['rakuten']['axes']]
    j, attr = next((j, a) for j, a in enumerate(sku['attributes']) if a['title'] == '本体縦幅')
    assert attr['value'] == '200' and attr['unit'] == 'cm'
    assert '140×205cm' in products[case['dossier_id']]['au']['title']
    quote = '【ダブル】（約）幅140×長さ205cm'
    start = raw_html.index(quote)
    pad_evidence.append({'case_id': cid, 'variant_id': sku['variantId'],
        'attribute_path': f'$.api.data.itemInfoSku.sku[{i}].attributes[{j}].value',
        'attribute_value': attr['value'], 'attribute_unit': attr['unit'],
        'html_quote': quote, 'start': start, 'end': start + len(quote),
        'offset_space': 'decoded original EUC-JP HTML, Unicode characters',
        'raw_file': source['raw_file'], 'raw_sha256': source['sha256']})

fabric_ids = sorted(cid for cid, tags in flags.items() if 'fabric_attribute_not_discriminative' in tags)
app, _, source = embedded_for(fabric_ids[0])
all_variants = app['api']['data']['itemInfoSku']['sku']
family = [(i, v) for i, v in enumerate(all_variants) if v['selectorValues'][:2] == ['フラットタイプ', 'シングル']]
fabric_evidence = []
for i, sku in family:
    j, attr = next((j, a) for j, a in enumerate(sku['attributes']) if a['title'] == '素材（生地・毛糸）')
    path = f'$.api.data.itemInfoSku.sku[{i}].attributes[{j}].value'
    assert resolve(app, path) == 'パイル・タオル'
    fabric_evidence.append({'variant_id': sku['variantId'], 'selectors': sku['selectorValues'],
        'attribute_path': path, 'attribute_value': attr['value'],
        'raw_file': source['raw_file'], 'raw_sha256': source['sha256']})
assert len(family) == 10
assert {cases[cid]['rakuten']['variant_id'] for cid in fabric_ids} <= {v['variantId'] for _, v in family}
report['new_source_findings'] = {'pad_dimension_evidence': pad_evidence,
    'mattress_all_raw_variant_count': len(all_variants), 'mattress_flat_single_subset_count': len(family),
    'mattress_non_discriminative_attribute_evidence': fabric_evidence,
    'affected_current_matched_case_count': len(pad_ids) + len(fabric_ids),
    'physical_product_mismatch_confirmed': False}
print(json.dumps({'pad_conflict_cases': len(pad_ids), 'fabric_cases': len(fabric_ids), 'fabric_subset': len(family), 'fabric_total_raw_variants': len(all_variants)}, ensure_ascii=False))


{"pad_conflict_cases": 4, "fabric_cases": 6, "fabric_subset": 10, "fabric_total_raw_variants": 95}


In [6]:
annotation = ROOT / '.lab-output/sku-real-luna-annotation-inputs-20261010-v3/cases.jsonl'
annotated = jsonl(annotation)
assert {c['case_id'] for c in annotated} == set(by_csv)
split_leak = defaultdict(set)
for case in annotated:
    split_leak[(case['group_id'], case['rakuten']['source']['source_sku_key'])].add(case['split'])
split_counts = {}
for split in ('dev', 'test'):
    selected = [c for c in annotated if c['split'] == split]
    # Descriptive category profile only; never a matching-system rule.
    curtains = [c for c in selected if 'カーテン' in products[c['dossier_id']]['au']['title']]
    split_counts[split] = {'case_count': len(selected), 'title_mentions_curtain': len(curtains),
                          'curtain_share': len(curtains) / len(selected)}
needle = '天板高さ50cmのシルバーには持ち手がありません'
exception_presence = {}
for version in ('v2', 'v3'):
    dossier = ROOT / f'.lab-output/sku-real-luna-annotation-inputs-20261010-{version}/dossiers/pair-2731f3cede6c0af8.json'
    data = json.loads(dossier.read_text())
    exception_presence[version] = needle in data['rakuten_product']['description']['individual_description_excerpt']
    report['input_sha256'][str(dossier.relative_to(ROOT))] = sha(dossier)
prod = products['pair-2731f3cede6c0af8']
exception_presence['current_generic_full_description'] = any(needle in r['text'] for r in prod['rakuten']['descriptions'])
assert exception_presence == {'v2': False, 'v3': True, 'current_generic_full_description': True}
report['scope_and_distribution'] = {'split_counts': split_counts,
    'same_group_rakuten_sku_dev_test_crossings': sum(len(s) > 1 for s in split_leak.values()),
    'unique_rakuten_source_skus_in_legacy': len({c['rakuten']['source']['source_sku_key'] for c in annotated}),
    'csv_covers_family_or_novel': False, 'independent_semantic_review_of_all_2409_cases': False,
    'wash_silver_exception_present_in_semantic_description': exception_presence,
    'category_definition': 'AU title contains カーテン, reporting only'}
report['input_sha256'][str(annotation.relative_to(ROOT))] = sha(annotation)
for path, expected in report['input_sha256'].items():
    assert sha(ROOT / path) == expected
OUTPUT.mkdir(parents=True, exist_ok=True)
with (OUTPUT / 'structural-audit.json').open('x', encoding='utf-8') as stream:
    json.dump(report, stream, ensure_ascii=False, indent=2)
    stream.write('\n')
with (OUTPUT / 'quality-sidecar.csv').open('x', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(sidecar[0]))
    writer.writeheader()
    writer.writerows(sidecar)
print(json.dumps({'audit_saved': str(OUTPUT), 'sidecar_rows': len(sidecar), 'split_counts': split_counts,
                  'wash_exception': exception_presence}, ensure_ascii=False, indent=2))


{
  "audit_saved": "/workspace/Searching_Experiment-model-gates-20261010/.lab-output/sku-second-opinion-quality-20261011-v1",
  "sidecar_rows": 1383,
  "split_counts": {
    "dev": {
      "case_count": 625,
      "title_mentions_curtain": 0,
      "curtain_share": 0.0
    },
    "test": {
      "case_count": 758,
      "title_mentions_curtain": 720,
      "curtain_share": 0.9498680738786279
    }
  },
  "wash_exception": {
    "v2": false,
    "v3": true,
    "current_generic_full_description": true
  }
}


監査は転記・結合・引用と指摘された仕様の確認です。フラグ45件は確定誤り数ではありません。
未フラグのケースも独立に正解確定していません。試作用機械ラベルと最終評価用goldを区別します。
ページ間の仕様差を許容する運用と、販売セット全体の厳密一致評価を混ぜません。
保持した2026-10-10時点の原本を確認しており、現行Webページの再取得・モデル再推論はしていません。
